# Search people

In [ ]:
import timelink
timelink.version

In [ ]:
from timelink.notebooks import TimelinkNotebook
import os


tlnb = TimelinkNotebook()

In [ ]:

person_ids = ['deh-jose-simoes']

### Search for people, show in Kleio notation

In [ ]:
from timelink.api.models import Person

show_only=10

with tlnb.db.session() as session:
    persons = session.query(Person).filter(Person.id.in_(person_ids)).all()

    for person in persons[:show_only]:
        print(person.to_kleio())
        print()

## Fetch an entity and display with template

- [ ] Segmentar o template em header, elements, linking, biography, contained entities
- [ ] Impementar os mesmos elementos que o to_kleio() aceita

Need to collect while looping:

- all relations origins for incoming
- all relations destinations for outgoing
- all inside id
- all contained entities


In [ ]:
from jinja2 import Environment, FileSystemLoader,TemplateNotFound
from IPython.display import Markdown, display

import timelink
from timelink.api.models import Person
from dehergne_util import (build_biography, relation_label, value_link,
                          type_link, generate_index_notes,
                          generate_type_summaries,
                          generate_wikidata_notes,
                          expand_with_identifications)
from biographical_note import generate_biographical_note
from copresence import export_copresence_network


# eid = 'deh-matteo-ricci'
# eid = 'crono-avulsos-his1-47'


env = Environment(loader=FileSystemLoader("../templates"))
# register the biography builder so templates can call build_biography(model)
env.globals['build_biography'] = build_biography
env.globals['relation_label'] = relation_label
env.globals['value_link'] = value_link
env.globals['type_link'] = type_link
env.globals['generate_biographical_note'] = generate_biographical_note
template_path = "markdown/base/"
# Set to True to also generate value__type index notes (one page per
# attribute/value pair, aggregating all persons sharing it).
generate_index = True
output_dir = "/Users/jrc/jrc-notes/05 Timelink/dehergne/"
index_dir = output_dir + "index/"  # where value__type notes are written

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

try:
    template_name = Person.__name__ + "_markdown.j2"
    template = env.get_template(template_path + template_name)
except TemplateNotFound:
    print(f"Template {template_name} not found. Using default template.")
    template = env.get_template(template_path + "entity_default_markdown.j2")

with tlnb.db.session() as session:
    persons = session.query(Person).all()
    # expand the set with record-linkage occurrences + real entities, so that
    # co-occurring referido stubs and merged REntity records get their own pages
    # and contribute to the value__type index notes.
    persons = expand_with_identifications(persons, session)
    for person in persons:
        markdown_content = template.render(model=person,
                                            util=timelink.kleio.utilities,
                                            follow_identification=True,
                                            db=session)
        # display(Markdown(str(markdown_content)))
        # print(markdown_content)
        output_file = f"{person.id}.md"
        output_path = os.path.join(output_dir, output_file)
        # print(f"Writing markdown to {output_path}")
        with open(output_path, 'w') as f:
            f.write(markdown_content)

# Generate value__type index notes (aggregating pages with backlinks)
if generate_index:
    written = generate_index_notes(persons, index_dir)
    summaries = generate_type_summaries(persons, index_dir)
    wd_pages = generate_wikidata_notes(persons, index_dir + "wikidata/")
    nnodes, nedges = export_copresence_network(
        persons, f"{output_dir}copresence.graphml")
    print(f"Wrote {len(written)} index notes, {len(summaries)} type summaries, "
          f"and {len(wd_pages)} wikidata pages to {index_dir}")
    print(f"Co-presence network: {nnodes} nodes, {nedges} edges "
          f"-> {output_dir}copresence.graphml")




## Real persons

In [ ]:
from timelink.api.models import REntity

output_dir = "/Users/jrc/jrc-notes/05 Timelink/dehergne/"

try:
    template_name = REntity.__name__ + "_markdown.j2"
    template = env.get_template(template_path + template_name)
except TemplateNotFound:
    print(f"Template {template_name} not found. Using default template.")
    template = env.get_template(template_path + "entity_default_markdown.j2")

with tlnb.db.session() as session:
    rpersons = session.query(REntity).all()
    for rperson in rpersons:
        markdown_content = template.render(model=rperson,
                                            util=timelink.kleio.utilities)
        #  display(Markdown(str(markdown_content)))
        # print(markdown_content)
        output_file = f"{rperson.id}.md"
        output_path = os.path.join(output_dir, output_file)
        # print(f"Writing markdown to {output_path}")
        with open(output_path, 'w') as f:
            f.write(markdown_content)

## Acts

In [ ]:
from timelink.api.models import Act

output_dir = "/Users/jrc/jrc-notes/05 Timelink/dehergne/"

template_name = Act.__name__ + "_markdown.j2"
try:
    template = env.get_template(template_path + template_name)
except TemplateNotFound:
    print(f"Template {template_name} not found. Using default template.")
    template = env.get_template(template_path + "entity_default_markdown.j2")

with tlnb.db.session() as session:
    acts = session.query(Act).all()
    for act in acts[:1]:
        markdown_content = template.render(model=act,
                                            util=timelink.kleio.utilities)
        display(Markdown(str(markdown_content)))
        # print(markdown_content)
        output_file = f"{act.id}.md"
        output_path = os.path.join(output_dir, output_file)
        print(f"Writing markdown to {output_path}")
        with open(output_path, 'w') as f:
            f.write(markdown_content)